### Data load

In [60]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional (recommended for quick EDA plots)
import seaborn as sns

from scipy import stats

In [61]:
df = pd.read_csv("property.csv", encoding='latin1')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

#### data clean up

In [62]:
df.duplicated().sum()

np.int64(0)

In [63]:
df = df.drop_duplicates()

In [64]:
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')

In [65]:
df.head(5)


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,2016-12-03,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,2016-02-04,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,2017-03-04,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,2017-03-04,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,2016-06-04,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


#### 1. For the suburb of Altona, it is postulated that a typical property sells for $$800,000.
##### Use the data at hand to test this assumption. 
##### Is the typical property price really $ 800,000 or has it increased? 
##### Use a significance level of 5%. 

In [66]:
df['Price'] = df['Price'].astype(int)

In [67]:
df1 = df[df['Suburb']=='Altona']

df1describe = df1['Price'].describe().astype(int)
df1describe

count         74
mean      834830
std       291546
min       391000
25%       622500
50%       773500
75%       972500
max      1780000
Name: Price, dtype: int64

###### Null hypothesis: property price is still $$ 800,000 at 5% significance
###### Alternative Hypo: property price is greater then $800,000
###### right tailed test with right side rejection

In [68]:
significance_level = 0.05
sCount = df1describe['count']
sMean = df1describe['mean']
sStd  = df1describe['std']
hMean = 800000

In [69]:
#calculate z value for the data
std_error = sStd / np.sqrt(sCount)
z_value = (sMean - hMean) / std_error

#calculate critical value for right tail above which for rejecting the hypothesis
t_critical = stats.t.ppf(1 - significance_level, sCount)

print(z_value)
print(t_critical)


1.0276902754662889
1.6657068927340233


###### Answer: Since z value is below than critical value, so Hypothesis not rejected. This means the mean has not increased. Thus, property price is still around 800,000 with 95% confidence.

#### 2. For the year 2016, is there any difference in the prices of properties sold in the summer months vs winter months? 
##### • Consider months from October till March as winter months and rest as summer months. 
##### • Use a significance level of 5%. 

###### Prepare filtered data

In [70]:
df2 = df[df['Date'].dt.year == 2016]
df2['season'] = np.where(df2['Date'].dt.month.isin([10,11,12,1,2,3]),'Winter','Summer')
df2stat = df2.groupby('season')['Price'].describe().astype(int)
df2stat

C:\Users\Jintu\AppData\Local\Temp\ipykernel_38396\434635411.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['season'] = np.where(df2['Date'].dt.month.isin([10,11,12,1,2,3]),'Winter','Summer')


,count,mean,std,min,25%,50%,75%,max
season,,,,,,,,
Summer,4036,1048054,621493,85000,623000,881000,1310000,6500000
Winter,2300,1116647,695498,216000,655000,931499,1371250,6250000


###### Null hypothesis: there is no significant difference in mean between Summer or winter prices
###### Alternate Hypo: there is significant difference in means
###### this is a two tail test  uding a two sample t-test

In [71]:
se1 = np.square(df2stat['std']['Summer'])/df2stat['count']['Summer']
se2 = np.square(df2stat['std']['Winter'])/df2stat['count']['Winter']
std_error = np.sqrt(se1+se2)
dfcal = np.square(se1+se2) / ((np.square(se1)/ (df2stat['count']['Summer']-1))+(np.square(se2)/ (df2stat['count']['Winter']-1)))
z_score = (df2stat['mean']['Summer'] - df2stat['mean']['Winter']) / std_error
print(z_score)

t_critical = stats.t.ppf(1 - significance_level/2, dfcal)
print(t_critical)

-3.9211110471916903
1.9605090184668614


##### Since Z value is beyound the boundary of critical condition on the negative left side in rejection area - We reject the Hypothesis and accept alternative hypothesis. Thus, there is significant different in sales price between Summer and Winter months.

#### 3. For the suburb of Abbotsford, what is the probability that out of 10 properties sold, 3 will not have a car parking space?
##### • Use the column car in the dataset. 
##### • Round off your answer to 3 decimal places. 

In [84]:
df3 = df[df['Suburb']=='Abbotsford']
df3['Car'].isna().sum()
df3['Car_parking'] = np.where(df3['Car'].isin([0]),'No','Yes')
df3stat = df3.groupby('Car_parking')['Car'].describe()
df3stat

C:\Users\Jintu\AppData\Local\Temp\ipykernel_38396\133080885.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['Car_parking'] = np.where(df3['Car'].isin([0]),'No','Yes')


,count,mean,std,min,25%,50%,75%,max
Car_parking,,,,,,,,
No,15.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
Yes,41.0,1.292683,0.460646,1.0,1.0,1.0,2.0,2.0


In [109]:
eCountSold = 10
eCountWoCarPark =3
probWoCarPark = df3stat.loc['No']['count'] / df3stat['count'].sum()

prob_binomal = stats.binom.pmf(eCountWoCarPark,eCountSold,probWoCarPark)
np.round(prob_binomal,3)

np.float64(0.26)

Result 26%

#### 4. In the suburb of Abbotsford, what are the chances of finding a property with 3 rooms? Round your answer to 3 decimal places. 

In [110]:
df4 = df[df['Suburb']=='Abbotsford']

df4['Rooms3Present'] = np.where(df4['Rooms'].isin([3]),'Yes','No')
df4stat = df4.groupby('Rooms3Present')['Type'].describe()
df4stat

C:\Users\Jintu\AppData\Local\Temp\ipykernel_38396\4146413615.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4['Rooms3Present'] = np.where(df4['Rooms'].isin([3]),'Yes','No')


,count,unique,top,freq
Rooms3Present,,,,
No,36,3,h,22
Yes,20,3,h,18


In [113]:
prob3Rooms = df4stat.loc['Yes']['count'] / df4stat['count'].sum()
np.round(prob3Rooms,3)

np.float64(0.357)

Result 35.7%

#### 5. In the suburb of Abbotsford, what are the chances of finding a property with 2 bathrooms? Round your answer to 3 decimal places.

In [114]:
df5 = df[df['Suburb']=='Abbotsford']

df5['Bathroom2Present'] = np.where(df5['Bathroom'].isin([2]),'Yes','No')
df5stat = df5.groupby('Bathroom2Present')['Type'].describe()
df5stat

C:\Users\Jintu\AppData\Local\Temp\ipykernel_38396\2506619165.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df5['Bathroom2Present'] = np.where(df5['Bathroom'].isin([2]),'Yes','No')


,count,unique,top,freq
Bathroom2Present,,,,
No,37,2,h,27
Yes,19,3,h,13


In [115]:
prob2Bathroom = df5stat.loc['Yes']['count'] / df5stat['count'].sum()
np.round(prob2Bathroom,3)

np.float64(0.339)

Result 33.9%

#### 6. One-Sample Hypothesis Test (Industry Pricing) A real estate firm claims that the average property price in Richmond is $1,000,000. Using the dataset, test whether the actual average price is significantly different from this claim at a 5% significance level. Clearly state: 
##### • Null and alternative hypotheses 
##### • Test statistic 
##### • p-value 
##### • Final business conclusion

In [118]:
df6 = df[df['Suburb']=='Richmond']

df6stat = df6['Price'].describe().astype(int)
df6stat

count        260
mean     1083564
std       522353
min       270000
25%       684000
50%      1078000
75%      1350000
max      3335000
Name: Price, dtype: int64

##### Null Hypothesis : The average property price is $$1,000,000
##### Alternative Hypo: The average property price is significantly different from $1,000,000
##### two tail test 

In [123]:
significance_level = 0.05
sCount = df6stat['count']
sMean = df6stat['mean']
sStd  = df6stat['std']
hMean = 1000000

#calculate z value for the data
std_error = sStd / np.sqrt(sCount)
z_value = (sMean - hMean) / std_error

#calculate critical value for two tail above which for rejecting the hypothesis
t_critical = stats.t.ppf(1 - significance_level, sCount)

#p-value of significance of z_value whether lesser than significace level which is good
p_value = 2 * (1 - stats.t.cdf(abs(z_value),sCount-1))

print(f"p-value {np.round(p_value,2)}")
print(f"z-value {np.round(z_value,2)}")
print(f"critical value {np.round(t_critical,2)}")

p-value0.01
z-value2.58
critical value 1.65


###### Altenative hypothesis is accepted since the z-value from test is outside the critical value and p-value is lesser than signficance value. Thus the average property price is significantly different from $1,000,000

#### 7. Independent Two-Sample T-Test (Feature Impact) 
##### Do properties with car parking sell at a higher average price than properties without car parking, across the entire dataset? Use a 5% significance level and justify: 
##### • Choice of test 
##### • Interpretation of p-value 
##### • Business implications for developers 

In [126]:
df7 = df
df7['Car_parking'] = np.where(df7['Car'].isin([0]),'No','Yes')
df7stat = df7.groupby('Car_parking')['Price'].describe().astype(int)
df7stat

,count,mean,std,min,25%,50%,75%,max
Car_parking,,,,,,,,
No,1026,1079088,513754,85000,760000,1001000,1325000,5700000
Yes,12554,1075405,648514,131000,644250,897000,1330000,9000000


###### Null hypothesis: The average price of property with and without car parking is less than or equal each other.
###### Alternative Hypo: The average price is higher than properties without the car parking.
###### right tail

In [128]:
se1 = np.square(df7stat['std']['Yes'])/df7stat['count']['Yes']
se2 = np.square(df7stat['std']['No'])/df7stat['count']['No']
std_error = np.sqrt(se1+se2)
dfcal = np.square(se1+se2) / ((np.square(se1)/ (df7stat['count']['Yes']-1))+(np.square(se2)/ (df7stat['count']['No']-1)))
z_score = (df7stat['mean']['Yes'] - df7stat['mean']['No']) / std_error
print(z_score)

t_critical = stats.t.ppf(1 - significance_level, dfcal)
print(t_critical)

-0.21599205539771774
1.646019835649737


###### Since the z value is within the critical boundary value we will accept the Null hypothesis saying the average price of property with and without car parking is less than or equal each other.